# 04 · Fase 4 — Ferramentas externas e RAG sobre índice existente

Rubricas **4.1 a 4.5**: ferramentas hardcoded vs semânticas, integração com APIs, text-to-query, retriever sobre índice FAISS já salvo e diagnóstico de falha de recuperação.

> **Sobre o modo offline.** O enunciado pede alternativas *dummy* para APIs e credenciais. Sem `OPENAI_API_KEY` (ou com `MODO_OFFLINE=true`), o projeto troca o GPT por um **modelo roteirizado** (`src/llm.py`: regras determinísticas que escolhem a ferramenta e formatam a observação), os embeddings por `DeterministicFakeEmbedding` e as APIs por **fixtures gravados** de chamadas reais (`fixtures/`). A estrutura das chains, do agente, da memória e da verificação é exatamente a mesma; só a "inteligência" muda. As saídas deste notebook foram geradas no modo indicado na célula de configuração.

**Revisão de 11/11/2026:** as saídas abaixo são evidências históricas online, anteriores aos novos middlewares de contexto/verificação e à persistência episódica em SQLite. A versão revisada foi validada por testes offline; estas saídas não representam uma reexecução online da revisão.


**Execução da revisão ApuraMandato:** saídas reexecutadas offline. O texto comparativo online anterior é histórico; os notebooks anteriores à reexecução foram preservados em `entregaveis/historico/`. A versão ampliada e os limites externos estão no notebook 06 e no relatório atual.


## 4.1 Ferramentas por tipo

| Ferramenta | Tipo | Lógica | Quando o agente usa |
|---|---|---|---|
| `calcular_estatisticas` | **hardcoded** (determinística) | `statistics` do Python | qualquer conta: soma, média, variação % |
| `consultar_ceap` | **text-to-query** | LLM gera SQL; guardrails (só SELECT, LIMIT forçado, auditoria) | médias, rankings, séries, comparações entre vários deputados |
| `consultar_regras_cota` | **semântica** (RAG) | retriever sobre FAISS do Ato da Mesa | o que a cota pode pagar, limites, prazos |
| `buscar_deputado`, `listar_despesas` | **API externa** (web API pública, síncrona) | `httpx` + paginação + cache em fixtures | resolver nomes, despesas por período |
| `dados_fornecedor` | **API externa** (web API pública) | Brasil API por CNPJ | identificar empresa |
| `gravar_apuracao`, `lembrar_apuracao` | **memória** (vector store) | embeddings + filtro por repórter | fechar/recuperar apurações |

In [1]:
# Configuração comum: roda a partir da raiz do repositório e mostra o modo de execução
import json
import os
import sys
from importlib.metadata import version
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
import warnings

warnings.filterwarnings("ignore")

from src import config, llm

print(
    "LangChain",
    version("langchain"),
    "| langchain-core",
    version("langchain-core"),
    "| LangGraph",
    version("langgraph"),
    "| LangSmith",
    version("langsmith"),
)
print(
    "Modo:",
    "OFFLINE (modelo dummy roteirizado + fixtures)" if llm.offline() else f"ONLINE ({config.MODELO_AGENTE})",
)

LangChain 1.4.9 | langchain-core 1.7.6 | LangGraph 1.2.20 | LangSmith 0.14.17
Modo: OFFLINE (modelo dummy roteirizado + fixtures)


In [2]:
from src import tools
from src.data import consultar_ceap, ferramenta_regras_cota
from src.memory import gravar_apuracao, lembrar_apuracao

for t in tools.toolkit_basico() + [
    consultar_ceap,
    ferramenta_regras_cota(),
    gravar_apuracao,
    lembrar_apuracao,
]:
    print(f"{t.name:22} args={list(t.args)}")

buscar_deputado        args=['nome', 'uf', 'ano']
listar_despesas        args=['id_deputado', 'ano', 'mes', 'categoria']
dados_fornecedor       args=['cnpj']
calcular_estatisticas  args=['valores', 'operacao']
consultar_ceap         args=['anos', 'query_sql']
consultar_regras_cota  args=['query']
gravar_apuracao        args=['resumo', 'reporter_id']
lembrar_apuracao       args=['consulta', 'reporter_id']


## 4.2 Integração com APIs

- **Web API pública (Câmara, dadosabertos.camara.leg.br/api/v2):** síncrona, sem autenticação, paginada em 100 itens; detalhe descoberto nos testes: o endpoint de despesas só responde com `idLegislatura` junto do ano (a ferramenta deriva a legislatura do ano).
- **Web API pública (Brasil API):** consulta de CNPJ, sem chave.
- **Camada de cache/fixtures:** toda resposta é gravada em `fixtures/` (chave = hash de URL+parâmetros); em `MODO_OFFLINE` as ferramentas leem só dali, o que serve de *mock* para avaliação sem rede.

Divergência observada e documentada: para Tabata Amaral em 2025 a API ao vivo devolve R$ 530.231,55 e o CSV anual baixado em 07/11/2026 soma R$ 504.994,08 (a diferença está em passagens aéreas e lançamentos recentes). Por isso o prompt obriga o agente a citar a fonte e a não misturar valores das duas numa mesma conta.

In [3]:
print(tools.dados_fornecedor.invoke({"cnpj": "10.951.191/0001-20"}))
r = json.loads(tools.listar_despesas.invoke({"id_deputado": 204534, "ano": 2025, "mes": 3}))
print("março/2025:", r["total_registros"], "lançamentos, total R$", r["total_valor"])

{"razao_social": "HUEVOS PRODUCAO E ASSESSORIA S/S LTDA", "situacao": "ATIVA", "cnae_principal": "Agências de notícias", "municipio": "SAO PAULO-SP", "abertura": "2009-06-25", "fonte": "BrasilAPI / Minha Receita", "consultado_em": "fixture gravada"}
março/2025: 54 lançamentos, total R$ 69601.24


## 4.3 Text-to-query com guardrails

O modelo recebe o schema e exemplos (`SCHEMA_CEAP`) e gera SQL; a ferramenta só executa `SELECT`, bloqueia palavras de escrita, força `LIMIT 200` e imprime a consulta (`[AUDITORIA SQL]`) para rastreabilidade.

In [4]:
from src.data import SCHEMA_CEAP

print(SCHEMA_CEAP[:700], "...")


Tabela SQLite 'despesas_ceap' (uma linha por despesa reembolsada pela cota):
  id_deputado INTEGER (mesmo id da API da Camara; NULL para liderancas partidarias),
  nome_parlamentar TEXT, partido TEXT, uf TEXT, categoria TEXT, especificacao TEXT,
  fornecedor TEXT, cnpj_cpf TEXT (somente digitos), valor REAL (liquido; negativo = estorno),
  ano INTEGER, mes INTEGER, data TEXT (AAAA-MM-DD)
Categorias (valores exatos de 'categoria'): 'COMBUSTÍVEIS E LUBRIFICANTES.', 'PASSAGEM AÉREA - SIGEPA',
  'MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE PARLAMENTAR', 'SERVIÇO DE TÁXI, PEDÁGIO E ESTACIONAMENTO',
  'DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.', 'TELEFONIA', 'FORNECIMENTO DE ALIMENTAÇÃO DO PARLAMENT ...


In [5]:
print(
    consultar_ceap.invoke(
        {
            "query_sql": "SELECT partido, ROUND(SUM(valor),2) AS total FROM despesas_ceap WHERE categoria LIKE '%COMBUST%' GROUP BY partido ORDER BY total DESC LIMIT 5"
        }
    )
)
print(consultar_ceap.invoke({"query_sql": "DROP TABLE despesas_ceap"}))
print(consultar_ceap.invoke({"query_sql": "SELECT * FROM despesas_ceap"})[:80], "...  <- LIMIT 200 aplicado")

[AUDITORIA SQL] SELECT partido, ROUND(SUM(valor),2) AS total FROM despesas_ceap WHERE categoria LIKE '%COMBUST%' GROUP BY partido ORDER BY total DESC LIMIT 5
5 registros: [{'partido': 'PL', 'total': 24856846.16}, {'partido': 'PT', 'total': 23111455.37}, {'partido': 'PP', 'total': 18061752.84}, {'partido': 'UNIÃO', 'total': 17315224.0}, {'partido': 'PSD', 'total': 15885931.81}]
Erro de seguranca: apenas consultas SELECT sao permitidas.
[AUDITORIA SQL] SELECT * FROM despesas_ceap LIMIT 200
200 registros: [{'id_deputado': None, 'nome_parlamentar': 'LIDERANÇA DO GOVERNO  ...  <- LIMIT 200 aplicado


In [6]:
from src.agent import montar_agente, trajetoria

agente = montar_agente()
est = agente.invoke(
    {"messages": [("user", "Qual a média de gasto com combustível por deputado do PSB em 2025?")]},
    config={"configurable": {"thread_id": "sql"}},
)
print([(p["ferramenta"], p["args"]) for p in trajetoria(est) if p["tipo"] == "decisao"])
print(est["messages"][-1].content[:300])

[AUDITORIA SQL] SELECT ROUND(AVG(t.total),2) AS media_por_deputado FROM (SELECT id_deputado, SUM(valor) AS total FROM despesas_ceap WHERE partido='PSB' AND ano=2025 AND categoria LIKE '%COMBUST%' GROUP BY id_deputado) t LIMIT 200
[('consultar_ceap', {'query_sql': "SELECT ROUND(AVG(t.total),2) AS media_por_deputado FROM (SELECT id_deputado, SUM(valor) AS total FROM despesas_ceap WHERE partido='PSB' AND ano=2025 AND categoria LIKE '%COMBUST%' GROUP BY id_deputado) t", 'anos': [2025]})]
Media por deputado: R$ 48.849,91 no período 2025. Fonte: CSV CEAP (banco local). Período: 2025.


## 4.4 Retriever sobre índice FAISS existente

O índice foi construído e salvo na fase 2 (`indices/ato_mesa`). Aqui ele é **carregado** com `load_local` e exposto ao agente por `create_retriever_tool`, sem reconstruir.

In [7]:
from src.data import carregar_indice

indice = carregar_indice()
print("vetores no índice:", indice.index.ntotal)
est = agente.invoke(
    {"messages": [("user", "Qual o prazo para prestar contas das despesas da cota?")]},
    config={"configurable": {"thread_id": "rag"}},
)
print([(p["ferramenta"], p["args"]) for p in trajetoria(est) if p["tipo"] == "decisao"])
print(est["messages"][-1].content[:400])

vetores no índice: 110
[('consultar_regras_cota', {'query': 'Qual o prazo para prestar contas das despesas da cota?'})]
Segundo o Ato da Mesa 43/2009 (trechos recuperados):
§ 5º A retirada de bilhete aéreo em nome de assessores exigirá o registro do fato em sistema informatizado próprio até o primeiro dia útil do mês seguinte ao da emissão de passagem. (Parágrafo com redação dada pelo Ato da Mesa nº 5, de 25/2/2015, com supressão da expressão “... e de cônjuge ou companheiro do deputado...” pelo Ato da Mesa nº 8, d


## 4.5 Diagnóstico de falha de recuperação e correção

**Ferramenta open-box:** `buscar_com_scores` devolve `chunk_id`, score L2 (menor = mais próximo) e trecho, para enxergar o que o retriever trouxe. **Caso de falha:** perguntas em linguagem leiga ("hotel em Brasília") não casam com o vocabulário da norma ("hospedagem", "Distrito Federal"). **Correção:** *query transformation*: reescrever a pergunta nos termos do Ato da Mesa antes de buscar (LLM no modo online; glossário determinístico no offline).

In [8]:
from src.rag import buscar_com_scores, diagnosticar

d = diagnosticar("A cota pode pagar hotel em Brasília?", k=3, termo_esperado="Distrito Federal")
print("pergunta :", d["pergunta"])
print("reescrita:", d["reescrita"])
print("\nANTES  (acertou=%s)" % d["acertou_antes"])
for r in d["antes"]:
    print(f"  [{r['chunk_id']}] L2={r['score_l2']}: {r['trecho'][:100]}")
print("\nDEPOIS (acertou=%s)" % d["acertou_depois"])
for r in d["depois"]:
    print(f"  [{r['chunk_id']}] L2={r['score_l2']}: {r['trecho'][:100]}")

pergunta : A cota pode pagar hotel em Brasília?
reescrita: a cota pode pagar hospedagem em distrito federal?

ANTES  (acertou=False)
  [CHK_013] L2=412.7531: a) locação de imóveis;  b) condomínio;  c) IPTU e seguro contra incêndio; (Alínea com redação dada p
  [CHK_028] L2=412.8146: § 5º O disposto na alínea “b” inciso VIII enseja reembolso mensal e inacumulável de até 50% do valor
  [CHK_061] L2=416.0599: Art. 7º Serão descontados automaticamente em folha de pagamento do Deputado os valores relativos aos

DEPOIS (acertou=False)
  [CHK_010] L2=424.2666: § 4º O adicional de cota previsto no parágrafo anterior será creditado após o recebimento da relação
  [CHK_011] L2=440.6867: § 5º A compensação de que trata o § 4º do art. 2º do Ato da Mesa nº 104, de 1988, dar-se-á mediante 
  [CHK_107] L2=440.8483: TSE: candidatura é identificada por eleição e sequencial. Partido e UF pertencem à candidatura histó


In [9]:
for r in buscar_com_scores("limite mensal da cota para deputado de SP", k=3):
    print(f"[{r['chunk_id']}] L2={r['score_l2']}: {r['trecho'][:110]}")

[CHK_013] L2=476.6189: a) locação de imóveis;  b) condomínio;  c) IPTU e seguro contra incêndio; (Alínea com redação dada pelo Ato da
[CHK_058] L2=479.6343: § 5º A retirada de bilhete aéreo em nome de assessores exigirá o registro do fato em sistema informatizado pró
[CHK_006] L2=482.611: a) Vice-Líder da Maioria e da Minoria; (Alínea acrescida pelo Ato da Mesa nº 122, de 6/10/2016, produzindo efe


**Interpretação.** No modo online, os scores L2 permitem ver se a reescrita aproximou a pergunta do chunk certo (o Anexo com a tabela de limites e o § sobre hospedagem no Distrito Federal). No modo offline os embeddings são determinísticos mas não semânticos, então a busca é essencialmente aleatória: é o que explica os "acertou=False" quando este notebook roda sem chave. O ponto da rubrica é o **método**: expor scores e proveniência, identificar o vocabulário como causa, transformar a consulta e medir de novo. A segunda correção prevista (se a reescrita não bastar) é indexar também a FAQ da cota, escrita em linguagem leiga, com metadado `fonte`.